In [1]:
%run Utilities

StatementMeta(, 2b158327-e533-45d9-9bc6-51925140408d, 3, Finished, Available, Finished, True)

In [2]:
from pyspark.sql.functions import col,count,when

StatementMeta(, 2b158327-e533-45d9-9bc6-51925140408d, 4, Finished, Available, Finished, False)

In [3]:
df_customers = read_parquet_df("abfss://my_workspace@onelake.dfs.fabric.microsoft.com/Ecommerce_Data.Lakehouse/Files/bronze/customers")
#display(df_customers)

df_customers = df_customers.selectExpr(
    "customer_id",
    "country",
    "CAST(age AS INT) as age",
    "gender",
    "membership_tier",
    "CAST(registration_date AS DATE) as registration_date",
    "CAST(total_orders AS INT) as total_orders",
    "CAST(total_spend_usd AS DOUBLE) as total_spend_usd",
    "CAST(avg_order_value_usd AS DOUBLE) as avg_order_value_usd",
    "CAST(days_since_last_purchase AS INT) as days_since_last_purchase",
    "preferred_category",
    "preferred_device",
    "preferred_payment_method",
    "acquisition_channel",
    "CAST(reviews_given AS INT) as reviews_given",
    "CAST(avg_review_score AS DOUBLE) as avg_review_score",
    "CAST(returns_made AS INT) as returns_made",
    "CAST(wishlist_items AS INT) as wishlist_items",
    "CASE WHEN newsletter_subscribed = '1' THEN true ELSE false END as newsletter_subscribed",
    "CASE WHEN churned = '1' THEN true ELSE false END as churned"
)

df_customers.printSchema()

StatementMeta(, 2b158327-e533-45d9-9bc6-51925140408d, 5, Finished, Available, Finished, False)

root
 |-- customer_id: string (nullable = true)
 |-- country: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- membership_tier: string (nullable = true)
 |-- registration_date: date (nullable = true)
 |-- total_orders: integer (nullable = true)
 |-- total_spend_usd: double (nullable = true)
 |-- avg_order_value_usd: double (nullable = true)
 |-- days_since_last_purchase: integer (nullable = true)
 |-- preferred_category: string (nullable = true)
 |-- preferred_device: string (nullable = true)
 |-- preferred_payment_method: string (nullable = true)
 |-- acquisition_channel: string (nullable = true)
 |-- reviews_given: integer (nullable = true)
 |-- avg_review_score: double (nullable = true)
 |-- returns_made: integer (nullable = true)
 |-- wishlist_items: integer (nullable = true)
 |-- newsletter_subscribed: boolean (nullable = false)
 |-- churned: boolean (nullable = false)



In [4]:
df_customers_clean = df_customers.dropna()

StatementMeta(, 2b158327-e533-45d9-9bc6-51925140408d, 6, Finished, Available, Finished, False)

In [5]:
# Check for nulls
total_nulls = df_customers_clean.select(
    [count(when(col(c).isNull(), 1)).alias(c) for c in df_customers_clean.columns]
).collect()[0]

if sum(total_nulls) == 0:
    print("NO NULLS found!")
else:
    print(f"Still has nulls in: {[c for c in df_customers_clean.columns if total_nulls[c] > 0]}")

# Remove duplicates
df_customers_clean = df_customers_clean.dropDuplicates(['customer_id'])



StatementMeta(, 2b158327-e533-45d9-9bc6-51925140408d, 7, Finished, Available, Finished, False)

NO NULLS found!


In [6]:
df_customers_clean.write.mode("overwrite").format("delta").save("abfss://my_workspace@onelake.dfs.fabric.microsoft.com/Ecommerce_Data.Lakehouse/Files/silver/customers")

print("Customers data saved to Silver!")

StatementMeta(, 2b158327-e533-45d9-9bc6-51925140408d, 8, Finished, Available, Finished, False)

Customers data saved to Silver!


In [7]:
df_customers_2 = read_parquet_df(
    "abfss://my_workspace@onelake.dfs.fabric.microsoft.com/Ecommerce_Data.Lakehouse/Files/bronze/siddharthpal18/Dataset/main/customers2.parquet"
)

display(df_customers_2)

StatementMeta(, 2b158327-e533-45d9-9bc6-51925140408d, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d4746106-accf-4679-8a19-0aaccac71410)

In [8]:
df_customers_2 = df_customers_2.selectExpr(
    "customer_id",
    "country",
    "CAST(age AS INT) as age",
    "gender",
    "membership_tier",
    "CAST(registration_date AS DATE) as registration_date",
    "CAST(total_orders AS INT) as total_orders",
    "CAST(total_spend_usd AS DOUBLE) as total_spend_usd",
    "CAST(avg_order_value_usd AS DOUBLE) as avg_order_value_usd",
    "CAST(days_since_last_purchase AS INT) as days_since_last_purchase",
    "preferred_category",
    "preferred_device",
    "preferred_payment_method",
    "acquisition_channel",
    "CAST(reviews_given AS INT) as reviews_given",
    "CAST(avg_review_score AS DOUBLE) as avg_review_score",
    "CAST(returns_made AS INT) as returns_made",
    "CAST(wishlist_items AS INT) as wishlist_items",
    "CASE WHEN newsletter_subscribed = '1' THEN true ELSE false END as newsletter_subscribed",
    "CASE WHEN churned = '1' THEN true ELSE false END as churned"
)



StatementMeta(, 2b158327-e533-45d9-9bc6-51925140408d, 10, Finished, Available, Finished, False)

In [9]:
df_customers_2 = df_customers_2.dropna()
# Check for nulls
total_nulls = df_customers_2.select(
    [count(when(col(c).isNull(), 1)).alias(c) for c in df_customers_2.columns]
).collect()[0]

if sum(total_nulls) == 0:
    print("NO NULLS found!")
else:
    print(f"Still has nulls in: {[c for c in df_customers_2.columns if total_nulls[c] > 0]}")

# Remove duplicates
df_customers_2 = df_customers_2.dropDuplicates(['customer_id'])

StatementMeta(, 2b158327-e533-45d9-9bc6-51925140408d, 11, Finished, Available, Finished, False)

NO NULLS found!


In [10]:
df_customers_2.write.mode("overwrite").format("delta").save("abfss://my_workspace@onelake.dfs.fabric.microsoft.com/Ecommerce_Data.Lakehouse/Files/silver/customers2")

print("Customers data saved to Silver!")

StatementMeta(, 2b158327-e533-45d9-9bc6-51925140408d, 12, Finished, Available, Finished, False)

Customers data saved to Silver!


In [11]:
## Read monthly revenue data file
df_monthly_revenue = read_parquet_df(path="abfss://my_workspace@onelake.dfs.fabric.microsoft.com/Ecommerce_Data.Lakehouse/Files/bronze/monthly_revenue")

df_monthly_revenue = df_monthly_revenue.selectExpr(
    "CAST(year AS INT) as year",
    "CAST(month AS INT) as month",
    "quarter",
    "CAST(orders AS INT) as orders",
    "CAST(revenue_usd AS DOUBLE) as revenue_usd",
    "CAST(avg_order_value AS DOUBLE) as avg_order_value",
    "CAST(avg_discount_pct AS DOUBLE) as avg_discount_pct",
    "CAST(return_rate AS DOUBLE) as return_rate",
    "CAST(unique_customers AS INT) as unique_customers",
    "CAST(new_customers AS INT) as new_customers"
)

df_monthly_revenue.printSchema()

StatementMeta(, 2b158327-e533-45d9-9bc6-51925140408d, 13, Finished, Available, Finished, False)

root
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- quarter: string (nullable = true)
 |-- orders: integer (nullable = true)
 |-- revenue_usd: double (nullable = true)
 |-- avg_order_value: double (nullable = true)
 |-- avg_discount_pct: double (nullable = true)
 |-- return_rate: double (nullable = true)
 |-- unique_customers: integer (nullable = true)
 |-- new_customers: integer (nullable = true)



In [12]:
df_revenue_clean = df_monthly_revenue.dropna()

StatementMeta(, 2b158327-e533-45d9-9bc6-51925140408d, 14, Finished, Available, Finished, False)

In [13]:
total_nulls = df_revenue_clean.select(
    [count(when(col(c).isNull(), 1)).alias(c) for c in df_revenue_clean.columns]
).collect()[0]

if sum(total_nulls) == 0:
    print("NO NULLS found!")
else:
    print(f"Still has nulls in: {[c for c in df_revenue_clean.columns if total_nulls[c] > 0]}")

# Remove duplicates

df_revenue_clean = df_revenue_clean.dropDuplicates(["year", "month"])

StatementMeta(, 2b158327-e533-45d9-9bc6-51925140408d, 15, Finished, Available, Finished, False)

NO NULLS found!


In [14]:
df_revenue_clean.write.mode("overwrite").format("delta").save("abfss://my_workspace@onelake.dfs.fabric.microsoft.com/Ecommerce_Data.Lakehouse/Files/silver/monthly_revenue")

print("Monthly revenue data saved to Silver!")

StatementMeta(, 2b158327-e533-45d9-9bc6-51925140408d, 16, Finished, Available, Finished, False)

Monthly revenue data saved to Silver!


In [16]:
df_orders = read_parquet_df(path="abfss://my_workspace@onelake.dfs.fabric.microsoft.com/Ecommerce_Data.Lakehouse/Files/bronze/orders")

#display(df_orders)
df_orders = df_orders.selectExpr(
    "order_id",
    "customer_id",
    "CAST(order_date AS DATE) as order_date",
    "CAST(year AS INT) as year",
    "CAST(month AS INT) as month",
    "quarter",
    "day_of_week",
    "product_name",
    "category",
    "CAST(unit_price_usd AS DOUBLE) as unit_price_usd",
    "CAST(quantity AS INT) as quantity",
    "CAST(subtotal_usd AS DOUBLE) as subtotal_usd",
    "CAST(discount_pct AS DOUBLE) as discount_pct",
    "CAST(discount_amount_usd AS DOUBLE) as discount_amount_usd",
    "CAST(shipping_fee_usd AS DOUBLE) as shipping_fee_usd",
    "CAST(tax_pct AS DOUBLE) as tax_pct",
    "CAST(tax_amount_usd AS DOUBLE) as tax_amount_usd",
    "CAST(total_amount_usd AS DOUBLE) as total_amount_usd",
    "payment_method",
    "device_used",
    "CAST(delivery_days AS INT) as delivery_days",
    "CAST(delivery_date AS DATE) as delivery_date",
    "order_status",
    "CASE WHEN returned = '1' THEN true ELSE false END as returned",
    "CASE WHEN customer_rating IS NULL THEN 'No' ELSE CAST(customer_rating AS STRING) END as customer_rating",
    "CAST(session_duration_minutes AS INT) as session_duration_minutes",
    "CAST(pages_viewed_before_purchase AS INT) as pages_viewed_before_purchase",
    "CASE WHEN is_repeat_customer = '1' THEN true ELSE false END as is_repeat_customer"
)





StatementMeta(, 2b158327-e533-45d9-9bc6-51925140408d, 18, Finished, Available, Finished, False)

In [17]:
# Check nulls

df_orders_clean = df_orders.dropna()

StatementMeta(, 2b158327-e533-45d9-9bc6-51925140408d, 19, Finished, Available, Finished, False)

In [18]:
df_orders_clean = df_orders_clean.dropDuplicates(["order_id"])

StatementMeta(, 2b158327-e533-45d9-9bc6-51925140408d, 20, Finished, Available, Finished, False)

In [19]:
df_orders_clean.write.mode("overwrite").format("delta").save("abfss://my_workspace@onelake.dfs.fabric.microsoft.com/Ecommerce_Data.Lakehouse/Files/silver/orders")

print("Orders data saved to Silver!")

StatementMeta(, 2b158327-e533-45d9-9bc6-51925140408d, 21, Finished, Available, Finished, False)

Orders data saved to Silver!


In [20]:
df_orders_2 = read_parquet_df(path="Files/bronze/siddharthpal18/Dataset/main/synthetic_orders.parquet")

#display(df_orders_2)
df_orders_2 = df_orders_2.selectExpr(
    "order_id",
    "customer_id",
    "CAST(order_date AS DATE) as order_date",
    "CAST(year AS INT) as year",
    "CAST(month AS INT) as month",
    "quarter",
    "day_of_week",
    "product_name",
    "category",
    "CAST(unit_price_usd AS DOUBLE) as unit_price_usd",
    "CAST(quantity AS INT) as quantity",
    "CAST(subtotal_usd AS DOUBLE) as subtotal_usd",
    "CAST(discount_pct AS DOUBLE) as discount_pct",
    "CAST(discount_amount_usd AS DOUBLE) as discount_amount_usd",
    "CAST(shipping_fee_usd AS DOUBLE) as shipping_fee_usd",
    "CAST(tax_pct AS DOUBLE) as tax_pct",
    "CAST(tax_amount_usd AS DOUBLE) as tax_amount_usd",
    "CAST(total_amount_usd AS DOUBLE) as total_amount_usd",
    "payment_method",
    "device_used",
    "CAST(delivery_days AS INT) as delivery_days",
    "CAST(delivery_date AS DATE) as delivery_date",
    "order_status",
    "CASE WHEN returned = '1' THEN true ELSE false END as returned",
    "CASE WHEN customer_rating IS NULL THEN 'No' ELSE CAST(customer_rating AS STRING) END as customer_rating",
    "CAST(session_duration_minutes AS INT) as session_duration_minutes",
    "CAST(pages_viewed_before_purchase AS INT) as pages_viewed_before_purchase",
    "CASE WHEN is_repeat_customer = '1' THEN true ELSE false END as is_repeat_customer"
)





StatementMeta(, 2b158327-e533-45d9-9bc6-51925140408d, 22, Finished, Available, Finished, False)

In [21]:
df_orders_2 = df_orders_2.dropna()
df_orders_2 = df_orders_2.dropDuplicates(["order_id"])

df_orders_2.write.mode("overwrite").format("delta").save("abfss://my_workspace@onelake.dfs.fabric.microsoft.com/Ecommerce_Data.Lakehouse/Files/silver/orders2")

print("Orders2 data saved to Silver!")

StatementMeta(, 2b158327-e533-45d9-9bc6-51925140408d, 23, Finished, Available, Finished, False)

Orders2 data saved to Silver!


In [22]:

df_products = read_parquet_df(path="abfss://my_workspace@onelake.dfs.fabric.microsoft.com/Ecommerce_Data.Lakehouse/Files/bronze/product_summary")

df_products = df_products.selectExpr(
    "category",
    "product_name",
    "CAST(total_orders AS INT) as total_orders",
    "CAST(total_revenue_usd AS DOUBLE) as total_revenue_usd",
    "CAST(avg_price AS DOUBLE) as avg_price",
    "CAST(avg_rating AS DOUBLE) as avg_rating",
    "CAST(return_rate AS DOUBLE) as return_rate",
    "CAST(avg_discount_pct AS DOUBLE) as avg_discount_pct",
    "CAST(avg_delivery_days AS INT) as avg_delivery_days"
)

df_products.printSchema()

StatementMeta(, 2b158327-e533-45d9-9bc6-51925140408d, 24, Finished, Available, Finished, False)

root
 |-- category: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- total_orders: integer (nullable = true)
 |-- total_revenue_usd: double (nullable = true)
 |-- avg_price: double (nullable = true)
 |-- avg_rating: double (nullable = true)
 |-- return_rate: double (nullable = true)
 |-- avg_discount_pct: double (nullable = true)
 |-- avg_delivery_days: integer (nullable = true)



In [23]:
df_products_clean = df_products.dropna()

StatementMeta(, 2b158327-e533-45d9-9bc6-51925140408d, 25, Finished, Available, Finished, False)

In [24]:
df_products_clean = df_products_clean.dropDuplicates(["product_name", "category"])

StatementMeta(, 2b158327-e533-45d9-9bc6-51925140408d, 26, Finished, Available, Finished, False)

In [25]:
df_products_clean.write.mode("overwrite").format("delta").save("abfss://my_workspace@onelake.dfs.fabric.microsoft.com/Ecommerce_Data.Lakehouse/Files/silver/Products")

print("Products data saved to Silver!")

StatementMeta(, 2b158327-e533-45d9-9bc6-51925140408d, 27, Finished, Available, Finished, True)

Products data saved to Silver!


In [ ]:
spark.catalog.clearCache()
spark.stop()